# Silver Layer

- ## Create Industry Dimension Table

In [0]:
CREATE OR REPLACE TABLE workspace.appalachia.silver_dim_industry (
    industry_code STRING,
    industry_name STRING
);

INSERT INTO workspace.appalachia.silver_dim_industry VALUES
    ('10', 'Total, All Industries'),
    ('11', 'Agriculture, Forestry, Fishing and Hunting'),
    ('21', 'Mining, Quarrying, and Oil and Gas Extraction'),
    ('22', 'Utilities'),
    ('23', 'Construction'),
    ('31-33', 'Manufacturing'),
    ('42', 'Wholesale Trade'),
    ('44-45', 'Retail Trade'),
    ('48-49', 'Transportation and Warehousing'),
    ('51', 'Information'),
    ('52', 'Finance and Insurance'),
    ('53', 'Real Estate and Rental and Leasing'),
    ('54', 'Professional, Scientific, and Technical Services'),
    ('55', 'Management of Companies and Enterprises'),
    ('56', 'Administrative and Support and Waste Management and Remediation Services'),
    ('61', 'Educational Services'),
    ('62', 'Health Care and Social Assistance'),
    ('71', 'Arts, Entertainment, and Recreation'),
    ('72', 'Accommodation and Food Services'),
    ('81', 'Other Services except Public Administration'),
    ('92', 'Public Administration'),
    ('99', 'Unclassified');

## Testing Silver Industry Dimension Table

In [0]:
SELECT *
FROM workspace.appalachia.silver_dim_industry
ORDER BY industry_code;

## Load Silver County Dimension Data 

In [0]:
%python

from pyspark.sql.types import (
    StructType, 
    StructField, 
    StringType
)

county_schema = StructType([
    StructField("county_fips", StringType(), True),
    StructField("state_name", StringType(), True),
    StructField("county_name", StringType(), True)
])
county_dim_df = (
    spark.read
    .option("header", True)
    .schema(county_schema)
    .csv("/Volumes/workspace/appalachia/source_system/Appalachian Counties Served by ARC.csv")

)

## Inspect Silver County Dimension Data

In [0]:
%python
county_dim_df.printSchema()
county_dim_df.show()

## Transform Silver County Dimension Table

In [0]:
%python
from pyspark.sql.functions import trim, col

county_dim_df = (
    county_dim_df
    .withColumn("county_name", trim(col("county_name")))
    .withColumn("state_name", trim(col("state_name")))
    )   

## Create Silver County Dimension Table

In [0]:
%python
county_dim_df.write.mode("overwrite").saveAsTable("workspace.appalachia.silver_county_dim")

> ## Testing Silver County Dimension Table

In [0]:
SELECT *
FROM workspace.appalachia.silver_county_dim
ORDER BY state_name;

## Silver Broadband Transform

In [0]:
%python
from pyspark.sql.functions import col, substring

silver_broadband_df = spark.table("workspace.appalachia.bronze_broadband")
silver_braodband_df = silver_broadband_df.withColumn(
    "county_fips",
    substring(col("tractcode"), 1, 5)
)

In [0]:
%python
silver_braodband_df.select("tractcode", "county_fips").show(10)


In [0]:
%python
silver_broadband_df = (silver_braodband_df
    .withColumnRenamed("tractcode", "census_tractcode")
    .withColumnRenamed("pcat_10x1", "broadband_access")
)

In [0]:
%python
silver_braodband_df = (silver_braodband_df
    .drop("pcat_all")
    )

## Inspect Transformed Silver Broadbands Data

In [0]:
%python
silver_braodband_df.printSchema()
silver_braodband_df.show(10)


## Load Silver Broadband Table

In [0]:
%python
silver_braodband_df.write.mode("overwrite").saveAsTable("workspace.appalachia.silver_broadband")

## Test Silver Broadband Table

In [0]:
SELECT *
FROM workspace.appalachia.silver_broadband
LIMIT 10

## Testing QCEW Pre Transformation

In [0]:
SELECT annual_avg_emplvl, 
    avg_annual_pay, 
    total_annual_wages,
    area_fips
FROM workspace.appalachia.bronze_qcew
WHERE (agglvl_code = 70 OR
    agglvl_code = 74)
    AND qtr = 'A'
ORDER BY RAND()
LIMIT 35

## Transforming QCEW Data

In [0]:
%python
silver_qcew_df =(
    spark.table("workspace.appalachia.bronze_qcew")
    .withColumnRenamed("area_fips", "county_fips")
    .select(
        "county_fips",
        "industry_code",
        "agglvl_code",
        "year",
        "qtr",
        "total_annual_wages",
        "taxable_annual_wages",
        "avg_annual_pay",
        "annual_avg_emplvl"
    )
)
silver_qcew_df.createOrReplaceTempView("silver_qcew_working")

In [0]:
CREATE OR REPLACE TABLE workspace.appalachia.silver_qcew AS
SELECT 
    county_fips,
    industry_code,
    agglvl_code,
    year,
    qtr,
    total_annual_wages
    taxable_annual_wages,
    avg_annual_pay
    annual_avg_emplvl
FROM silver_qcew_working
WHERE (agglvl_code = 70 or agglvl_code = 74)
    AND qtr = 'A'

In [0]:
SELECT *
FROM workspace.appalachia.silver_qcew
LIMIT 10


## Transforming Population Data

In [0]:
%python
from pyspark.sql.functions import col, concat, lpad

silver_population_df =(
    spark.table("workspace.appalachia.bronze_population")
    .withColumn("county", col("COUNTY").cast("string"))
    .withColumn("state", col("STATE").cast("string"))
    .withColumn("county_fips", 
        concat(
            lpad(col("state"), 2, "0"),
            lpad(col("county"), 3, "0"))
))

In [0]:
%python
from pyspark.sql.functions import col, regexp_extract

pop_est_df = (
    silver_population_df
    .select(
        "county_fips",
        "POPESTIMATE2020",
        "POPESTIMATE2021",
        "POPESTIMATE2022",
        "POPESTIMATE2023",
        "POPESTIMATE2024",
        "POPESTIMATE2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["POPESTIMATE2020",
                    "POPESTIMATE2021",
                    "POPESTIMATE2022",
                    "POPESTIMATE2023",
                    "POPESTIMATE2024",
                    "POPESTIMATE2025"],
            variableColumnName="year",
            valueColumnName="population"
        )
    )

pop_est_df = pop_est_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int")
)
    

In [0]:
%python
pop_est_df.show(20)

In [0]:
%python
from pyspark.sql.functions import col, regexp_extract

numeric_change_pop_df = (
    silver_population_df
    .select(
        "county_fips",
        "NPOPCHG2020",
        "NPOPCHG2021",
        "NPOPCHG2022",
        "NPOPCHG2023",
        "NPOPCHG2024",
        "NPOPCHG2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["NPOPCHG2020",
                    "NPOPCHG2021",
                    "NPOPCHG2022",
                    "NPOPCHG2023",
                    "NPOPCHG2024",
                    "NPOPCHG2025"],
            variableColumnName="year",
            valueColumnName="numeric_change_pop"
        )
    )
numeric_change_pop_df = numeric_change_pop_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
numeric_change_pop_df.show(20)

In [0]:
%python
working_df = pop_est_df.join(
    numeric_change_pop_df,
    on=["county_fips", "year"],
    how="inner"
)

In [0]:
%python
working_df.show(20)

In [0]:
%python
births_df = (
    silver_population_df
    .select(
        "county_fips",
        "BIRTHS2020",
        "BIRTHS2021",
        "BIRTHS2022",
        "BIRTHS2023",
        "BIRTHS2024",
        "BIRTHS2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["BIRTHS2020",
                    "BIRTHS2021",
                    "BIRTHS2022",
                    "BIRTHS2023",
                    "BIRTHS2024",
                    "BIRTHS2025"],
            variableColumnName="year",
            valueColumnName="births"
    )
)

births_df = births_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))


In [0]:
%python
births_df.show(20)

In [0]:
%python
working_df = working_df.join(
    births_df,
    on=["county_fips", "year"],
    how="inner"
)

In [0]:
%python
working_df.show(20)

In [0]:
%python
deaths_df = (
    silver_population_df
    .select(
        "county_fips",
        "DEATHS2020",
        "DEATHS2021",
        "DEATHS2022",
        "DEATHS2023",
        "DEATHS2024",
        "DEATHS2025"
        )
    .unpivot(
        ids=["county_fips"],
        values=["DEATHS2020",
            "DEATHS2021",
            "DEATHS2022",
            "DEATHS2023",
            "DEATHS2024",
            "DEATHS2025"],
        variableColumnName="year",
        valueColumnName="deaths"
        )
)

deaths_df = deaths_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))
       


In [0]:
%python
deaths_df.show(20)

In [0]:
%python
working_df = working_df.join(
    deaths_df,
    on=["county_fips", "year"],
    how="inner"
)


In [0]:
%python
working_df.show(20)

In [0]:
%python
natural_change_df = (
    silver_population_df
    .select(
        "county_fips",
        "NATURALCHG2020",
        "NATURALCHG2021",
        "NATURALCHG2022",
        "NATURALCHG2023",
        "NATURALCHG2024",
        "NATURALCHG2025"
    )
    .unpivot(
        ids=["county_fips"],
        values=["NATURALCHG2020",
                "NATURALCHG2021",
                "NATURALCHG2022",
                "NATURALCHG2023",
                "NATURALCHG2024",
                "NATURALCHG2025"],
        variableColumnName="year",
        valueColumnName="natural_change"
    )
)

natural_change_df = natural_change_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int")
)

In [0]:
%python
natural_change_df.show(10)

In [0]:
%python
int_mig_df = (
    silver_population_df
    .select(
        "county_fips",
        "INTERNATIONALMIG2020",
        "INTERNATIONALMIG2021",
        "INTERNATIONALMIG2022",
        "INTERNATIONALMIG2023",
        "INTERNATIONALMIG2024",
        "INTERNATIONALMIG2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["INTERNATIONALMIG2020",
                    "INTERNATIONALMIG2021",
                    "INTERNATIONALMIG2022",
                    "INTERNATIONALMIG2023",
                    "INTERNATIONALMIG2024",
                    "INTERNATIONALMIG2025"],
            variableColumnName="year",
            valueColumnName="internat_mig"
        )
    )

int_mig_df = int_mig_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
int_mig_df.show(10)

In [0]:
%python
dom_mig_df = (
    silver_population_df
    .select(
        "county_fips",
        "DOMESTICMIG2020",
        "DOMESTICMIG2021",
        "DOMESTICMIG2022",
        "DOMESTICMIG2023",
        "DOMESTICMIG2024",
        "DOMESTICMIG2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["DOMESTICMIG2020",
                    "DOMESTICMIG2021",
                    "DOMESTICMIG2022",
                    "DOMESTICMIG2023",
                    "DOMESTICMIG2024",
                    "DOMESTICMIG2025"],
            variableColumnName="year",
            valueColumnName="dom_mig"
        )
    )

dom_mig_df = dom_mig_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
dom_mig_df.show(10)

In [0]:
%python
net_mig_df = (
    silver_population_df
    .select(
        "county_fips",
        "NETMIG2020",
        "NETMIG2021",
        "NETMIG2022",
        "NETMIG2023",
        "NETMIG2024",
        "NETMIG2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["NETMIG2020",
                    "NETMIG2021",
                    "NETMIG2022",
                    "NETMIG2023",
                    "NETMIG2024",
                    "NETMIG2025"],
            variableColumnName="year",
            valueColumnName="net_mig"
        )
    )

net_mig_df = net_mig_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
net_mig_df.show(10)

In [0]:
%python
residual_df = (
    silver_population_df
    .select(
        "county_fips",
        "RESIDUAL2020",
        "RESIDUAL2021",
        "RESIDUAL2022",
        "RESIDUAL2023",
        "RESIDUAL2024",
        "RESIDUAL2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["RESIDUAL2020",
                    "RESIDUAL2021",
                    "RESIDUAL2022",
                    "RESIDUAL2023",
                    "RESIDUAL2024",
                    "RESIDUAL2025"],
            variableColumnName="year",
            valueColumnName="residual"
        )
    )

residual_df = residual_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
residual_df.show(10)

In [0]:
%python
gq_est_df = (
    silver_population_df
    .select(
        "county_fips",
        "GQESTIMATES2020",
        "GQESTIMATES2021",
        "GQESTIMATES2022",
        "GQESTIMATES2023",
        "GQESTIMATES2024",
        "GQESTIMATES2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["GQESTIMATES2020",
                    "GQESTIMATES2021",
                    "GQESTIMATES2022",
                    "GQESTIMATES2023",
                    "GQESTIMATES2024",
                    "GQESTIMATES2025"],
            variableColumnName="year",
            valueColumnName="gq_est"
        )
    )

gq_est_df = gq_est_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
gq_est_df.show(10)

In [0]:
%python
birth_rates_df = (
    silver_population_df
    .select(
        "county_fips",
        "RBIRTH2021",
        "RBIRTH2022",
        "RBIRTH2023",
        "RBIRTH2024",
        "RBIRTH2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["RBIRTH2021",
                    "RBIRTH2022",
                    "RBIRTH2023",
                    "RBIRTH2024",
                    "RBIRTH2025"],
            variableColumnName="year",
            valueColumnName="birth_rates"
        )
    )

birth_rates_df = birth_rates_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
birth_rates_df.show(10)

In [0]:
%python
death_rates_df = (
    silver_population_df
    .select(
        "county_fips",
        "RDEATH2021",
        "RDEATH2022",
        "RDEATH2023",
        "RDEATH2024",
        "RDEATH2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["RDEATH2021",
                    "RDEATH2022",
                    "RDEATH2023",
                    "RDEATH2024",
                    "RDEATH2025"],
            variableColumnName="year",
            valueColumnName="death_rates"
        )
    )

death_rates_df = death_rates_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
death_rates_df.show(10)

In [0]:
%python
nat_change_rates_df = (
    silver_population_df
    .select(
        "county_fips",
        "RNATURALCHG2021",
        "RNATURALCHG2022",
        "RNATURALCHG2023",
        "RNATURALCHG2024",
        "RNATURALCHG2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["RNATURALCHG2021",
                    "RNATURALCHG2022",
                    "RNATURALCHG2023",
                    "RNATURALCHG2024",
                    "RNATURALCHG2025"],
            variableColumnName="year",
            valueColumnName="nat_change_rates"
        )
    )

nat_change_rates_df = nat_change_rates_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
nat_change_rates_df.show(10)

In [0]:
%python
dom_mig_rates_df = (
    silver_population_df
    .select(
        "county_fips",
        "RDOMESTICMIG2021",
        "RDOMESTICMIG2022",
        "RDOMESTICMIG2023",
        "RDOMESTICMIG2024",
        "RDOMESTICMIG2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["RDOMESTICMIG2021",
                    "RDOMESTICMIG2022",
                    "RDOMESTICMIG2023",
                    "RDOMESTICMIG2024",
                    "RDOMESTICMIG2025"],
            variableColumnName="year",
            valueColumnName="dom_mig_rates"
        )
    )

dom_mig_rates_df = dom_mig_rates_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
dom_mig_rates_df.show(10)

In [0]:
%python
int_mig_rates_df = (
    silver_population_df
    .select(
        "county_fips",
        "RINTERNATIONALMIG2021",
        "RINTERNATIONALMIG2022",
        "RINTERNATIONALMIG2023",
        "RINTERNATIONALMIG2024",
        "RINTERNATIONALMIG2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["RINTERNATIONALMIG2021",
                    "RINTERNATIONALMIG2022",
                    "RINTERNATIONALMIG2023",
                    "RINTERNATIONALMIG2024",
                    "RINTERNATIONALMIG2025"],
            variableColumnName="year",
            valueColumnName="int_mig_rates"
        )
    )

int_mig_rates_df = int_mig_rates_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
net_mig_rates_df = (
    silver_population_df
    .select(
        "county_fips",
        "RNETMIG2021",
        "RNETMIG2022",
        "RNETMIG2023",
        "RNETMIG2024",
        "RNETMIG2025"
        )
    .unpivot(
            ids=["county_fips"],
            values=["RNETMIG2021",
                    "RNETMIG2022",
                    "RNETMIG2023",
                    "RNETMIG2024",
                    "RNETMIG2025"],
            variableColumnName="year",
            valueColumnName="net_mig_rates"
        )
    )

net_mig_rates_df = net_mig_rates_df.withColumn(
    "year",
    regexp_extract(col("year"), r"(\d{4})", 1).cast("int"))

In [0]:
%python
working_df = (
    working_df
    .join(
        natural_change_df,
        on=["county_fips", "year"],
        how="inner"
    )
    .join(
        int_mig_df,
        on=["county_fips", "year"],
        how="inner"
    )
    .join(
        dom_mig_df,
        on=["county_fips", "year"],
        how="inner"
    )
)

In [0]:
%python
working_df = (
    working_df
    .join(
        net_mig_df,
        on=["county_fips", "year"],
        how="inner"
    )
    .join(
        residual_df,
        on=["county_fips", "year"],
        how="inner"
    )
    .join(
        gq_est_df,
        on=["county_fips", "year"],
        how="inner"
    )
)

In [0]:
%python
working_df.show(10)

In [0]:
%python
working_df = (
    working_df
    .join(
        birth_rates_df,
        on=["county_fips", "year"],
        how="inner"
    )
    .join(
        death_rates_df,
        on=["county_fips", "year"],
        how="inner"
    )
    .join(
        nat_change_rates_df,
        on=["county_fips", "year"],
        how="inner"
    )
)

In [0]:
%python
working_df = (
    working_df
    .join(
        dom_mig_rates_df,
        on=["county_fips", "year"],
        how="inner"
    )
    .join(
        int_mig_rates_df,
        on=["county_fips", "year"],
        how="inner"
    )
    .join(
        net_mig_rates_df,
        on=["county_fips", "year"],
        how="inner"
    )
)

In [0]:
%python
silver_population_df = working_df

In [0]:
%python
silver_population_df.show(20)

In [0]:
%python
silver_population_df.write.mode("overwrite").saveAsTable("workspace.appalachia.silver_population")

In [0]:
ALTER TABLE workspace.appalachia.silver_dim_industry RENAME TO workspace.appalachia.silver_industry_dim